In [2]:
# Instalação das bibliotecas no ambiente virtual ativo
!pip install tensorflow matplotlib numpy pillow

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.4.1-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
  Using cached grpcio-1.84.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached keras


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"Versão do TensorFlow: {tf.__version__}")

Versão do TensorFlow: 2.21.0


In [3]:
# Parâmetros de entrada das imagens
IMG_HEIGHT = 180
IMG_WIDTH = 180
BATCH_SIZE = 32

# Caminho para o diretório de dados (ajuste se necessário)
DATA_DIR = os.path.join(os.getcwd(), 'database')

In [6]:
# Conjunto de treinamento (80% das imagens)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
)

# Conjunto de validação (20% das imagens)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print(f"Classes identificadas: {class_names}")  # Exemplo: ['aprovadas', 'reprovadas']

Found 32 files belonging to 2 classes.


Using 26 files for training.
Found 32 files belonging to 2 classes.
Using 6 files for validation.
Classes identificadas: ['aprovadas', 'reprovadas']


In [8]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [9]:
num_classes = len(class_names)

model = keras.Sequential(
    [
        layers.Rescaling(
            1.0 / 255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
        ),  # Normalização dos pixels
        layers.Conv2D(16, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"],
)

model.summary()

c:\Users\User\projeto-integrador-4termo\venv\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 180, 180, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 90, 90, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 90, 90, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 45, 45, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,965,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,988,898 (15.22 MB)

 Trainable params: 3,988,898 (15.22 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
epochs = 10

history = model.fit(train_ds, validation_data=val_ds, epochs=epochs)

Epoch 1/10


c:\Users\User\projeto-integrador-4termo\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.1923 - loss: 0.8207 - val_accuracy: 0.5000 - val_loss: 5.4484
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step - accuracy: 0.5000 - loss: 4.2433 - val_accuracy: 1.0000 - val_loss: 0.5957
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step - accuracy: 1.0000 - loss: 0.4174 - val_accuracy: 0.5000 - val_loss: 1.3085
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.5000 - loss: 1.3023 - val_accuracy: 0.5000 - val_loss: 0.6914
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.5000 - loss: 0.6818 - val_accuracy: 1.0000 - val_loss: 0.5141
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 1.0000 - loss: 0.3673 - val_accuracy: 0.5000 - val_loss: 0.8975
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.8077 - loss: 0.4386 - val_accuracy: 0.5000 - val_loss: 0.7323
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step - accuracy: 0.8077 - loss: 0.3678 - val_accuracy: 1.0000 - val_loss: 0.3916
Epoch 

In [11]:
# 1. Converte o modelo Keras para TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# 2. Salva o arquivo .tflite no diretório corrente (ia-modelo)
tflite_model_path = "modelo_inspecao.tflite"
with open(tflite_model_path, "wb") as f:
  f.write(tflite_model)

print(
    f"Modelo TensorFlow Lite salvo com sucesso em: {os.path.abspath(tflite_model_path)}"
)

INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmpfb75sv0a\assets


INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmpfb75sv0a\assets


Saved artifact at 'C:\Users\User\AppData\Local\Temp\tmpfb75sv0a'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 180, 180, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  1934287838032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934287839376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934287837840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934287838992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934286053136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934286052752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934286053904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934286054672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934286053712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1934286049680: TensorSpec(shape=(), dtype=tf.resource, name=None)
Modelo T

In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# Caminho do modelo salvo no passo de treinamento
tflite_model_path = "modelo_inspecao.tflite"

# Inicializa o interpretador do TensorFlow Lite
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

# Obtém os detalhes das camadas de entrada e saída
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Modelo TFLite carregado com sucesso!")

Modelo TFLite carregado com sucesso!


c:\Users\User\projeto-integrador-4termo\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [13]:
def preprocess_image(img_path):
    # As dimensões devem ser idênticas às do treinamento (180x180)
    img = image.load_img(img_path, target_size=(180, 180))
    
    # Converte a imagem para um array numpy
    img_array = image.img_to_array(img)
    
    # Adiciona a dimensão do batch (1, 180, 180, 3)
    img_array = np.expand_dims(img_array, axis=0)
    
    return img_array

In [21]:
# Caminho para uma imagem de teste real simulando a câmera da esteira
caminho_imagem_teste = "C:/Users/User/projeto-integrador-4termo/ia-modelo/database/aprovadas/aprovada_branca - Copia (4).jpg"
# Pré-processa a imagem
input_data = preprocess_image(caminho_imagem_teste)

# Insere a imagem no tensor de entrada do modelo
interpreter.set_tensor(input_details[0]['index'], input_data)

# Executa a inferência
interpreter.invoke()

# Extrai o resultado do tensor de saída
output_data = interpreter.get_tensor(output_details[0]['index'])

# A saída é um array de probabilidades (Softmax). Pega o índice com maior probabilidade
predicted_class_index = np.argmax(output_data)
probabilidade = np.max(output_data)

# Mapeia o índice de volta para o nome da classe (Aprovada ou Reprovada)
# Certifique-se de que a ordem das classes corresponde a `train_ds.class_names`
classes = ['aprovadas', 'reprovadas'] 
resultado_final = classes[predicted_class_index]

print(f"Status da Peça: {resultado_final.upper()}")
print(f"Grau de Confiança: {probabilidade * 100:.2f}%")

if resultado_final == 'reprovadas':
    print("AÇÃO: Sinalizar CLP via OPC-UA para acionar CILINDRO DE SIMPLES AÇÃO (Desviar peça).")
else:
    print("AÇÃO: Sinalizar CLP via OPC-UA para continuar movimentação via CILINDRO DE DUPLA AÇÃO (Aprovada).")

Status da Peça: APROVADAS
Grau de Confiança: 75.62%
AÇÃO: Sinalizar CLP via OPC-UA para continuar movimentação via CILINDRO DE DUPLA AÇÃO (Aprovada).
